# ELIA WILD — controlled Kaggle launch

This notebook validates the installed organism first, then runs one real Qwen-backed cognitive cycle and exports an authenticated continuity checkpoint.

**Scope:** first controlled GPU proof. Do not treat one successful run as proof of long-horizon autonomy.


In [ ]:
REPO_REF = 'elia/genesis-1.7.1-consolidation'
!git clone --branch {REPO_REF} --single-branch https://github.com/vvseweedno/ELIA-WILD.git
%cd ELIA-WILD
!python -m pip install -q -e '.[test]'


In [ ]:
# Keep runtime state outside the checkout so code can be replaced without deleting identity state.
import os
os.environ['ELIA_STATE_DIR'] = '/kaggle/working/elia-state'

# Optional but strongly recommended: add ELIA_CHECKPOINT_KEY in Kaggle Secrets.
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('ELIA_CHECKPOINT_KEY')
    if key:
        os.environ['ELIA_CHECKPOINT_KEY'] = key
except Exception as exc:
    print('Checkpoint secret not loaded:', type(exc).__name__)

os.environ['ELIA_AUTO_CHECKPOINT_PATH'] = '/kaggle/working/elia-genesis.eliacp'


In [ ]:
# CPU-only proof path. This must be green before any model is loaded.
!elia-doctor
!elia-bootstrap --cycles 2
!elia-vitals
!python -m elia --verify
!python -m elia --status
!elia-supervisor --dry-run


In [ ]:
# Confirm the accelerator before spending GPU quota.
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing.'
print(torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1024**3:.2f} GiB')


In [ ]:
# Install the pinned 4-bit Qwen backend only after CPU validation succeeds.
!python -m pip install -q -e '.[gpu]'


In [ ]:
# One real cognitive cycle. --force-wake only bypasses a future scheduler timestamp;
# integrity, identity and budget guards remain active.
!python -m elia --force-wake --cycles 1


In [ ]:
# Verify the accepted state immediately after cognition.
!python -m elia --verify
!elia-vitals
!python -m elia --status
!tail -n 8 /kaggle/working/elia-state/chronicle.jsonl


In [ ]:
# Export the handoff checkpoint explicitly. The printed digest is a trust anchor, not a secret.
assert os.environ.get('ELIA_CHECKPOINT_KEY'), 'Set ELIA_CHECKPOINT_KEY in Kaggle Secrets before exporting continuity state.'
!python -m elia --checkpoint-export /kaggle/working/elia-genesis.eliacp
!ls -lh /kaggle/working/elia-genesis.eliacp


## After the first successful GPU cycle

Preserve `/kaggle/working/elia-genesis.eliacp` in a **private** persistence channel and keep the printed checkpoint digest separately. The current checkpoint format authenticates integrity but is not yet an encrypted confidentiality envelope, so do not place sensitive private memory in a public Dataset.

The automated `scripts/kaggle_wake.py` relay can be enabled only after the controlled proof succeeds and its state Dataset/kernel identifiers are configured.
